# Wave Equation and C Code Generation

Turn the scalar-wave right-hand sides into NRPy symbolic expressions and complete C assignments.

Navigation: [Index](../index.ipynb) | Previous: [Finite-Difference Playground](../2-numerical_methods/finite_difference_playground.ipynb) | Next: [Start-to-Finish Cartesian Wave Project](start_to_finish_wave_cartesian.ipynb)


## Symbolic Wave Right-Hand Sides
The first-order scalar wave system uses `uu_t = vv` and `vv_t = wavespeed**2` times the spatial Laplacian of `uu`.

## Import SymPy for Residual Checks

These imports expose the NRPy and Python tools used in the next steps.


In [ ]:
import sympy as sp


## Import Wave Equation and Codegen Tools

These imports expose the NRPy registries and infrastructure writers used below.


In [ ]:
import nrpy.c_codegen as ccg
import nrpy.grid as grid
import nrpy.indexedexp as ixp
import nrpy.params as par
from nrpy.equations.wave_equation.WaveEquation_RHSs import WaveEquation_RHSs
from nrpy.equations.wave_equation.WaveEquation_Solutions_InitialData import (
    WaveEquation_solution_Cartesian,
)


## Step 3: Build the Cartesian wave-equation RHSs

The printed right-hand sides are the symbolic expressions later passed to C code generation.

In [ ]:
for name in ["uu", "vv", "uu_rhs", "vv_rhs"]:
    grid.glb_gridfcs_dict.pop(name, None)
par.set_parval_from_str("Infrastructure", "BHaH")
rhs = WaveEquation_RHSs()
print("uu_rhs =", rhs.uu_rhs)
print("vv_rhs =", rhs.vv_rhs)


## Compare NRPy RHSs with the Hand Formula

A zero residual confirms that the symbolic expression matches the expected identity.


In [ ]:
wavespeed = sp.Symbol("wavespeed", real=True)
vv = sp.Symbol("vv", real=True)
uu_dDD = ixp.declarerank2("uu_dDD", symmetry="sym01")
hand_uu_rhs = vv
hand_vv_rhs = wavespeed**2 * sum(uu_dDD[i][i] for i in range(3))
print("uu residual:", sp.simplify(rhs.uu_rhs - hand_uu_rhs))
print("vv residual:", sp.simplify(rhs.vv_rhs - hand_vv_rhs))
if (
    sp.simplify(rhs.uu_rhs - hand_uu_rhs) != 0
    or sp.simplify(rhs.vv_rhs - hand_vv_rhs) != 0
):
    raise RuntimeError("Expected the NRPy RHSs to match the hand-written RHSs.")


## Emit RHS Assignments

The output shows how a symbolic expression is emitted as C.


In [ ]:
print("complete RHS assignments:")
print(
    ccg.c_codegen(
        [rhs.uu_rhs, rhs.vv_rhs],
        ["uu_rhs", "vv_rhs"],
        include_braces=False,
        verbose=False,
    )
)


## Emit Exact-Solution Assignments

The output shows how a symbolic expression is emitted as C.


In [ ]:
plane_wave = WaveEquation_solution_Cartesian(WaveType="PlaneWave")
print("complete exact-solution assignments:")
print(
    ccg.c_codegen(
        [plane_wave.uu_exactsoln, plane_wave.vv_exactsoln],
        ["uu_exact", "vv_exact"],
        include_braces=False,
        verbose=False,
    )
)


The residuals confirm that the generated symbolic right-hand sides match the hand-written wave system. The C assignments are the compact kernel body used by complete projects.


## Continue to Cartesian Project Generation
- [C Code Generation](../1-intro/c_codegen.ipynb)
- [Finite Differences](../1-intro/finite_difference.ipynb)
- [Start-to-Finish Cartesian Wave Project](start_to_finish_wave_cartesian.ipynb)
